# 03f · Rețeaua de canale din OpenStreetMap
Descarcă `waterway` = `canal`, `ditch`, `drain` din zonă și calculează distanța de la fiecare parcelă la canale și la toată rețeaua.

- Rezultate: `data/out/canals_osm.gpkg` (EPSG:3844, doar partea românească) și `data/out/parcel_canals.parquet` (`dist_canal_m`, `dist_network_m`).
- Sursa: © OpenStreetMap contributors (ODbL), de menționat pe hărțile și slide-urile care o folosesc.
- OSM nu e o evidență oficială a sistemelor de irigații: rețeaua poate fi incompletă și nu spune dacă un canal a avut apă în 2024.

In [ ]:
import os, sys
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import overpass_ways_to_gdf, canal_distances
import geopandas as gpd, requests, ee
from shapely.geometry import shape
ee.Initialize(project=GEE_PROJECT)
BBOX = ee.Geometry.Rectangle(list(AOI_BBOX))
RO = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Romania'))
aoi_poly = gpd.GeoSeries([shape(BBOX.intersection(RO.geometry(), ee.ErrorMargin(10)).getInfo())], crs=4326).to_crs(CRS).iloc[0]

In [ ]:
west, south, east, north = AOI_BBOX
query = f'[out:json][timeout:120];(way["waterway"~"^(canal|drain|ditch)$"]({south},{west},{north},{east}););out geom;'
r = requests.post('https://overpass-api.de/api/interpreter', data={'data': query}, headers={'User-Agent': 'gisday2026/1.0'}, timeout=240)
r.raise_for_status()
ways = overpass_ways_to_gdf(r.json()['elements']).to_crs(CRS)
ways = ways[ways.intersects(aoi_poly)].copy()
ways['geometry'] = ways.intersection(aoi_poly)
ways['km'] = ways.length / 1000
ways.to_file(OUT / 'canals_osm.gpkg', layer='canale', driver='GPKG')
print('segmente:', len(ways), '| km pe tip:', ways.groupby('waterway').km.sum().round().to_dict(), '| usage:', ways.usage.value_counts().to_dict())

In [ ]:
parcels = gpd.read_parquet(OUT / 'parcels.parquet')
dist = canal_distances(parcels, ways)
dist.to_parquet(OUT / 'parcel_canals.parquet')
assert len(dist) == len(parcels)
print(dist[['dist_canal_m', 'dist_network_m']].describe(percentiles=[.5, .95]).round().to_string())